# Silver Layer: Wikipedia Pageview Cleaning

This notebook transforms raw Wikimedia pageview records from the Bronze layer into a cleaned, analytics-ready Silver Delta table.

## What This Notebook Does

The Silver layer:

- creates proper timestamp and date columns
- identifies desktop and mobile access
- converts page titles into a more readable format
- removes records without positive pageviews
- removes empty or symbol-only titles
- excludes the Wikipedia Main Page
- removes technical Wikimedia namespaces
- keeps the columns required for downstream analytics

## Input Table

`workspace.wiki_bronze.pageviews_raw`

## Output Table

`workspace.wiki_silver.pageviews_clean`

## Why This Step Matters

The Bronze table remains close to the original Wikimedia source. The Silver layer improves data quality and creates a consistent structure for analysis.

The resulting table contains article-level pageview records that can be categorized and aggregated in the Gold layer.

## Current Scope

The Silver table is rebuilt from the rolling twelve-hour Bronze snapshot during each pipeline run.

In [0]:
# Use the project catalog

spark.sql("USE CATALOG workspace")

BRONZE_TABLE = "wiki_bronze.pageviews_raw"
SILVER_TABLE = "wiki_silver.pageviews_clean"

TECHNICAL_NAMESPACE_PATTERN = (
    r"^(Media|Special|Talk|"
    r"User(?:_talk)?|"
    r"Wikipedia(?:_talk)?|"
    r"File(?:_talk)?|"
    r"MediaWiki(?:_talk)?|"
    r"Template(?:_talk)?|"
    r"Help(?:_talk)?|"
    r"Category(?:_talk)?|"
    r"Portal(?:_talk)?|"
    r"Draft(?:_talk)?|"
    r"TimedText(?:_talk)?|"
    r"Module(?:_talk)?|"
    r"Gadget(?:_talk)?|"
    r"Gadget_definition(?:_talk)?|"
    r"Education_Program|Topic|Event):"
)

print("Using catalog: workspace")
print("Bronze table:", BRONZE_TABLE)
print("Silver table:", SILVER_TABLE)

Using catalog: workspace
Bronze table: wiki_bronze.pageviews_raw
Silver table: wiki_silver.pageviews_clean


In [0]:
from pyspark.sql import functions as F

silver_df = (
    spark.table(BRONZE_TABLE)

    # Create a real timestamp from source_date and source_hour
    .withColumn(
        "view_timestamp",
        F.to_timestamp(
            F.concat(
                F.col("source_date"),
                F.lit(" "),
                F.format_string("%02d:00:00", F.col("source_hour"))
            )
        )
    )

    # Create a proper date column for daily aggregations later
    .withColumn(
        "view_date",
        F.to_date(F.col("source_date"))
    )

    # Convert project code into a readable access method
    .withColumn(
        "access_method",
        F.when(F.col("project") == "en", "desktop")
         .when(F.col("project") == "en.m", "mobile")
         .otherwise("other")
    )

    # Make Wikipedia titles easier to read
    .withColumn(
        "page_title_clean",
        F.regexp_replace(F.col("page_title"), "_", " ")
    )

    # Keep only positive pageviews
    .filter(F.col("views") > 0)

    # Remove empty or whitespace-only titles
    .filter(F.length(F.trim(F.col("page_title_clean"))) > 0)

    # Remove titles that contain only symbols such as "-", "!!!", quotes, etc.
    # Keep only titles with at least one letter or number.
    .filter(F.col("page_title_clean").rlike(r"[\p{L}\p{N}]"))

    # Remove the Wikipedia Main Page because it dominates but is not a topic/article insight
    .filter(F.col("page_title") != "Main_Page")

    # Remove technical Wikimedia namespaces
    .filter(
    ~F.col("page_title").rlike(TECHNICAL_NAMESPACE_PATTERN)
    )

    # Keep only the columns needed for analytics
    .select(
        "view_timestamp",
        "view_date",
        "source_date",
        "source_hour",
        "project",
        "access_method",
        "page_title",
        "page_title_clean",
        "views",
        "source_url"
    )
)

display(silver_df.orderBy(F.desc("views")).limit(20))

view_timestamp,view_date,source_date,source_hour,project,access_method,page_title,page_title_clean,views,source_url
2026-06-14T18:00:00.000Z,2026-06-14,2026-06-14,18,en.m,mobile,Curaçao,Curaçao,384782,https://dumps.wikimedia.org/other/pageviews/2026/2026-06/pageviews-20260614-180000.gz
2026-06-14T21:00:00.000Z,2026-06-14,2026-06-14,21,en.m,mobile,Zion_Suzuki,Zion Suzuki,292982,https://dumps.wikimedia.org/other/pageviews/2026/2026-06/pageviews-20260614-210000.gz
2026-06-14T20:00:00.000Z,2026-06-14,2026-06-14,20,en.m,mobile,Oliver_Tree,Oliver Tree,285495,https://dumps.wikimedia.org/other/pageviews/2026/2026-06/pageviews-20260614-200000.gz
2026-06-14T21:00:00.000Z,2026-06-14,2026-06-14,21,en.m,mobile,Oliver_Tree,Oliver Tree,249686,https://dumps.wikimedia.org/other/pageviews/2026/2026-06/pageviews-20260614-210000.gz
2026-06-14T19:00:00.000Z,2026-06-14,2026-06-14,19,en.m,mobile,Oliver_Tree,Oliver Tree,225638,https://dumps.wikimedia.org/other/pageviews/2026/2026-06/pageviews-20260614-190000.gz
2026-06-14T19:00:00.000Z,2026-06-14,2026-06-14,19,en.m,mobile,Curaçao,Curaçao,218892,https://dumps.wikimedia.org/other/pageviews/2026/2026-06/pageviews-20260614-190000.gz
2026-06-14T22:00:00.000Z,2026-06-14,2026-06-14,22,en.m,mobile,Oliver_Tree,Oliver Tree,207934,https://dumps.wikimedia.org/other/pageviews/2026/2026-06/pageviews-20260614-220000.gz
2026-06-14T22:00:00.000Z,2026-06-14,2026-06-14,22,en.m,mobile,Zion_Suzuki,Zion Suzuki,163071,https://dumps.wikimedia.org/other/pageviews/2026/2026-06/pageviews-20260614-220000.gz
2026-06-14T23:00:00.000Z,2026-06-14,2026-06-14,23,en.m,mobile,Oliver_Tree,Oliver Tree,160781,https://dumps.wikimedia.org/other/pageviews/2026/2026-06/pageviews-20260614-230000.gz
2026-06-15T03:00:00.000Z,2026-06-15,2026-06-15,3,en.m,mobile,Yasin_Ayari,Yasin Ayari,134463,https://dumps.wikimedia.org/other/pageviews/2026/2026-06/pageviews-20260615-030000.gz


In [0]:
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(SILVER_TABLE)

print(f"Silver table saved: {SILVER_TABLE}")

Silver table saved: wiki_silver.pageviews_clean


In [0]:
spark.sql("""
SELECT
  access_method,
  COUNT(*) AS rows,
  SUM(views) AS total_views
FROM wiki_silver.pageviews_clean
GROUP BY access_method
ORDER BY access_method
""").show()

+-------------+--------+-----------+
|access_method|    rows|total_views|
+-------------+--------+-----------+
|      desktop|11413124|   34360494|
|       mobile|14000872|   92450984|
+-------------+--------+-----------+



In [0]:
spark.sql("""
SELECT
  page_title_clean,
  SUM(views) AS total_views
FROM wiki_silver.pageviews_clean
GROUP BY page_title_clean
ORDER BY total_views DESC
LIMIT 20
""").show(truncate=False)

+-------------------------------------+-----------+
|page_title_clean                     |total_views|
+-------------------------------------+-----------+
|Oliver Tree                          |1998806    |
|Curaçao                              |998089     |
|Zion Suzuki                          |658224     |
|2026 FIFA World Cup                  |461923     |
|UFC Freedom 250                      |412611     |
|2026 Rio de Janeiro mid-air collision|330362     |
|Gaspi                                |323495     |
|Jalen Brunson                        |318264     |
|Curaçao national football team       |256259     |
|Yasin Ayari                          |235882     |
|FIFA World Cup                       |183037     |
|Dick Advocaat                        |171576     |
|Nathaniel Brown (footballer)         |165500     |
|Josh Hokit                           |162896     |
|Ilia Topuria                         |160520     |
|Lucas A. Vignale                     |152515     |
|Felix Nmech

In [0]:
display(
    spark.table("wiki_silver.pageviews_clean")
    .orderBy(F.desc("views"))
    .limit(20)
)

view_timestamp,view_date,source_date,source_hour,project,access_method,page_title,page_title_clean,views,source_url
2026-06-14T18:00:00.000Z,2026-06-14,2026-06-14,18,en.m,mobile,Curaçao,Curaçao,384782,https://dumps.wikimedia.org/other/pageviews/2026/2026-06/pageviews-20260614-180000.gz
2026-06-14T21:00:00.000Z,2026-06-14,2026-06-14,21,en.m,mobile,Zion_Suzuki,Zion Suzuki,292982,https://dumps.wikimedia.org/other/pageviews/2026/2026-06/pageviews-20260614-210000.gz
2026-06-14T20:00:00.000Z,2026-06-14,2026-06-14,20,en.m,mobile,Oliver_Tree,Oliver Tree,285495,https://dumps.wikimedia.org/other/pageviews/2026/2026-06/pageviews-20260614-200000.gz
2026-06-14T21:00:00.000Z,2026-06-14,2026-06-14,21,en.m,mobile,Oliver_Tree,Oliver Tree,249686,https://dumps.wikimedia.org/other/pageviews/2026/2026-06/pageviews-20260614-210000.gz
2026-06-14T19:00:00.000Z,2026-06-14,2026-06-14,19,en.m,mobile,Oliver_Tree,Oliver Tree,225638,https://dumps.wikimedia.org/other/pageviews/2026/2026-06/pageviews-20260614-190000.gz
2026-06-14T19:00:00.000Z,2026-06-14,2026-06-14,19,en.m,mobile,Curaçao,Curaçao,218892,https://dumps.wikimedia.org/other/pageviews/2026/2026-06/pageviews-20260614-190000.gz
2026-06-14T22:00:00.000Z,2026-06-14,2026-06-14,22,en.m,mobile,Oliver_Tree,Oliver Tree,207934,https://dumps.wikimedia.org/other/pageviews/2026/2026-06/pageviews-20260614-220000.gz
2026-06-14T22:00:00.000Z,2026-06-14,2026-06-14,22,en.m,mobile,Zion_Suzuki,Zion Suzuki,163071,https://dumps.wikimedia.org/other/pageviews/2026/2026-06/pageviews-20260614-220000.gz
2026-06-14T23:00:00.000Z,2026-06-14,2026-06-14,23,en.m,mobile,Oliver_Tree,Oliver Tree,160781,https://dumps.wikimedia.org/other/pageviews/2026/2026-06/pageviews-20260614-230000.gz
2026-06-15T03:00:00.000Z,2026-06-15,2026-06-15,3,en.m,mobile,Yasin_Ayari,Yasin Ayari,134463,https://dumps.wikimedia.org/other/pageviews/2026/2026-06/pageviews-20260615-030000.gz


In [0]:
spark.sql("""
SELECT
  page_title_clean,
  SUM(views) AS total_views
FROM wiki_silver.pageviews_clean
WHERE NOT page_title_clean RLIKE '[\\\\p{L}\\\\p{N}]'
GROUP BY page_title_clean
ORDER BY total_views DESC
LIMIT 20
""").show(truncate=False)

+----------------+-----------+
|page_title_clean|total_views|
+----------------+-----------+
+----------------+-----------+



In [0]:
(
    spark.table(SILVER_TABLE)
    .filter(F.col("page_title").rlike(TECHNICAL_NAMESPACE_PATTERN))
    .groupBy("page_title_clean")
    .agg(F.sum("views").alias("total_views"))
    .orderBy(F.desc("total_views"))
    .limit(20)
    .show(truncate=False)
)

+----------------+-----------+
|page_title_clean|total_views|
+----------------+-----------+
+----------------+-----------+

